In [1]:
%load_ext autoreload
%autoreload 2

import sys
sys.path.append("../balance_metrics")

import yaml
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots


data_path = "../../experiment_data/balance_metrics/cross_layer_random.csv"

random_data = pd.read_csv(data_path)

with open('../across-blocks/layer_orders_cross_layer.yml', 'r') as f:
    layer_order = yaml.safe_load(f)
        
import matplotlib.pyplot as plt

import plotly.io as pio
pio.renderers.default = "vscode"
pio.templates.default = "ggplot2"

color_seq = px.colors.qualitative.Dark24
color_seq_train = px.colors.qualitative.Pastel
color_seq_val = px.colors.qualitative.Set1

In [2]:
full_data = pd.concat([random_data], ignore_index=True)
full_data = full_data.sort_values(["dataset"], ascending=True)

In [3]:
resnet = full_data[(full_data["model"].str.contains("resnet")) & (full_data["split"] == "train")].copy()

resnet["layer_idx"] = resnet["layer"].apply(lambda x: layer_order['resnet'][x]['idx'])
resnet["layer_names"] = resnet["layer"].apply(lambda x: layer_order['resnet'][x]['name'])
resnet.sort_values("layer_idx", inplace=True)

resnet_accs = resnet[(resnet["layer"] == "a1") & (resnet["dataset"].str.contains("-r"))][["dataset", "model", "train_acc", "val_acc", "sackin_index"]]
resnet_accs["random_prop"] = resnet_accs["dataset"].apply(lambda x: float(x.split("-r")[1]))
resnet_accs.sort_values("random_prop", inplace=True)

fig = make_subplots()
for i, row in resnet_accs.iterrows():
    color = color_seq[i % len(color_seq)]
    fig.add_trace(go.Scatter(x=[row["random_prop"], row["random_prop"]], y=[row["train_acc"], row["val_acc"]], mode="lines+markers", line = dict(color=color), name=f"{row['model']} ({row['random_prop']})"))
    
fig.update_annotations(font_size=16)
fig.update_layout(margin=dict(l=0, r=0, t=0, b=0), width=220*2, height=180*2, font=dict(size=14), showlegend=False)
fig.update_xaxes(title_text="Randomness Proportion", title_standoff=18, automargin=True, nticks=10)
fig.update_yaxes(title_text="Accuracy", range=[0.0, 1.05], nticks=10, title_standoff=18, automargin=True)
fig.show()
# fig.write_image("acc-gap.png", scale=8)

In [4]:
penultimate_accs = full_data[(full_data["layer"] == "a1") & (full_data["dataset"].str.contains("-r")) & (full_data["split"] == "train")][["dataset", "model", "train_acc", "val_acc", "sackin_index"]]
penultimate_accs["random_prop"] = penultimate_accs["dataset"].apply(lambda x: float(x.split("-r")[1]))
penultimate_accs.sort_values("random_prop", inplace=True)

fig = px.line(penultimate_accs, x="random_prop", y="sackin_index", color="model")

fig.update_annotations(font_size=16)
fig.update_layout(margin=dict(l=0, r=0, t=0, b=0), width=220*2, height=180*2, font=dict(size=14))
fig.update_legends(yanchor="top", xanchor="right", y=0.27, x = 0.999, bgcolor="rgba(0,0,0,0)")
fig.update_xaxes(title_text="Randomness Proportion", title_standoff=18, automargin=True, nticks=10)
fig.update_yaxes(title_text="Sackin Index", title_standoff=18, automargin=True)
fig.show()
# fig.write_image("imbalance-gain.png", scale=8)

In [5]:
resnet["rprop"] = resnet["dataset"].apply(lambda x: float(x.split("-r")[-1]))
resnet2 = resnet[resnet['rprop'].isin([0.0, 0.2, 0.4, 0.6, 0.8, 1.0])].copy()

resnet2["Random Proportion"] = resnet2["rprop"]

resnet2 = resnet2.sort_values(by=["rprop", "model"], ascending=False)

fig = px.line(resnet2, x="layer_names", y="sackin_index", color="Random Proportion", color_discrete_sequence=px.colors.qualitative.Plotly, title="ResNet Sackin")

fig.update_annotations(font_size=16)
fig.update_layout(margin=dict(l=0, r=0, t=0, b=0), width=440*2, height=180*2, font=dict(size=14), showlegend=False)
fig.update_xaxes(title_text="", automargin=True, tickangle=80)
fig.update_yaxes(title_text="Sackin Index", title_standoff=18, automargin=True)
fig.show()
# fig.write_image("resnet-mnist-random-blocks.png", scale=8)

In [7]:
resnet["rprop"] = resnet["dataset"].apply(lambda x: float(x.split("-r")[-1]))
resnet2 = resnet[resnet['rprop'].isin([0.0, 0.05, 0.1, 0.15, 0.2])].copy()

resnet2["Randomness"] = resnet2["rprop"]

resnet2 = resnet2.sort_values(by=["rprop", "model"], ascending=False)

fig = px.line(resnet2, x="layer_names", y="sackin_index", color="Randomness", color_discrete_sequence=px.colors.qualitative.Plotly, title="ResNet Sackin")

fig.update_layout(margin=dict(l=0, r=0, t=0, b=0), width=440*2, height=180*2, font=dict(size=24), legend=dict(
	xanchor="right", yanchor="top", x=1.1, y=0.98, bgcolor="rgba(255,255,255,0.5)", font=dict(size=16)))
fig.update_xaxes(title_text="", automargin=True, tickangle=0)
fig.update_yaxes(title_text="Sackin Index", title_standoff=18, automargin=True)
# fig.show()
fig.write_image("resnet-mnist-random-blocks-more-train.png", scale=8)